# Advanced `collections.ChainMap` Problems — With Solutions

This notebook is an advanced practice set on Python's `collections.ChainMap`.

## What you will practice

- lookup precedence and key collisions
- mutation, insertion, and deletion semantics
- reference/view behavior
- `new_child()`, `parents`, and `maps`
- flattening and materialization
- provenance: finding which mapping supplied a value
- custom write/delete policies
- scoped symbol tables
- transactional overlays and rollback
- immutable parent mappings
- nested `ChainMap` structures
- practical configuration layering
- lightweight benchmarking and trade-off analysis

## Best-practice workflow

For each problem:

1. Read the prompt.
2. Predict the behavior before running code.
3. Write your own solution.
4. Compare with the provided solution.
5. Run the assertions.
6. Modify the inputs and test edge cases.

> Important mental model: `ChainMap` is a **view over mappings**, searched from left to right. Standard writes and deletes target only the **first mapping**.


In [1]:
from collections import ChainMap
from collections.abc import MutableMapping
from types import MappingProxyType
from pprint import pprint
import timeit

print("Imports ready.")


Imports ready.


## Warm-up: precedence, views, and mutation

Before the advanced problems, verify the three rules that drive almost everything else:

- lookups search mappings from left to right;
- the mappings are held by reference, not copied;
- normal writes/deletes apply only to the first mapping.


In [2]:
base = {"host": "prod.example.com", "port": 5432, "debug": False}
user = {"debug": True}
session = {}

cfg = ChainMap(session, user, base)

assert cfg["debug"] is True
assert cfg["host"] == "prod.example.com"

session["host"] = "localhost"
assert cfg["host"] == "localhost"

cfg["port"] = 15432
assert session["port"] == 15432
assert base["port"] == 5432

print("Effective configuration:", dict(cfg))
print("Underlying maps:")
pprint(cfg.maps)


Effective configuration: {'host': 'localhost', 'port': 15432, 'debug': True}
Underlying maps:
[{'host': 'localhost', 'port': 15432},
 {'debug': True},
 {'debug': False, 'host': 'prod.example.com', 'port': 5432}]


# Problem 1 — Multi-layer configuration with strict precedence

You have four configuration layers:

1. request-specific overrides
2. user preferences
3. environment configuration
4. application defaults

Construct a `ChainMap` so that the first matching key wins.

Then compute:

- the effective `timeout`
- the effective `theme`
- the effective `region`
- the effective `retries`

Finally, update `timeout` through the `ChainMap` and prove that only the first mapping is changed.

### Data

```python
defaults = {"timeout": 30, "theme": "light", "region": "us-east", "retries": 3}
environment = {"timeout": 20, "region": "eu-west"}
user = {"theme": "dark"}
request = {"timeout": 5}
```


## Solution 1

The highest-priority mapping must come first. Therefore the chain order is:

`request -> user -> environment -> defaults`

A standard assignment such as `config["timeout"] = 7` writes into `request`, even if the visible value originally came from a deeper mapping.


In [3]:
defaults = {"timeout": 30, "theme": "light", "region": "us-east", "retries": 3}
environment = {"timeout": 20, "region": "eu-west"}
user = {"theme": "dark"}
request = {"timeout": 5}

config = ChainMap(request, user, environment, defaults)

assert config["timeout"] == 5
assert config["theme"] == "dark"
assert config["region"] == "eu-west"
assert config["retries"] == 3

config["timeout"] = 7

assert request["timeout"] == 7
assert environment["timeout"] == 20
assert defaults["timeout"] == 30

print(dict(config))
pprint(config.maps)


{'timeout': 7, 'theme': 'dark', 'region': 'eu-west', 'retries': 3}
[{'timeout': 7},
 {'theme': 'dark'},
 {'region': 'eu-west', 'timeout': 20},
 {'region': 'us-east', 'retries': 3, 'theme': 'light', 'timeout': 30}]


# Problem 2 — The "update" trap

Consider:

```python
primary = {"a": 1}
secondary = {"b": 2, "shared": "secondary"}
cm = ChainMap(primary, secondary)
```

Predict the exact contents of both dictionaries after:

```python
cm["b"] = 200
cm["shared"] = "new"
cm["c"] = 300
```

Why is this different from "find the key and update the dictionary where it already exists"?


## Solution 2

`ChainMap.__setitem__` does not search for an existing key. It always writes to `maps[0]`.

So all three assignments land in `primary`. This may **shadow** values that still remain unchanged in `secondary`.


In [4]:
primary = {"a": 1}
secondary = {"b": 2, "shared": "secondary"}
cm = ChainMap(primary, secondary)

cm["b"] = 200
cm["shared"] = "new"
cm["c"] = 300

assert primary == {
    "a": 1,
    "b": 200,
    "shared": "new",
    "c": 300,
}
assert secondary == {
    "b": 2,
    "shared": "secondary",
}

assert cm["b"] == 200
assert cm["shared"] == "new"

print("primary:", primary)
print("secondary:", secondary)
print("effective:", dict(cm))


primary: {'a': 1, 'b': 200, 'shared': 'new', 'c': 300}
secondary: {'b': 2, 'shared': 'secondary'}
effective: {'b': 200, 'shared': 'new', 'a': 1, 'c': 300}


# Problem 3 — Delete to reveal a hidden parent value

A key exists in both the child and parent mapping:

```python
child = {"mode": "test", "x": 1}
parent = {"mode": "prod", "y": 2}
cm = ChainMap(child, parent)
```

1. What is `cm["mode"]` initially?
2. Delete `cm["mode"]`.
3. What is `cm["mode"]` afterward?
4. Explain why the key still exists in the chain.
5. Try deleting `"y"` through the `ChainMap`. What happens, and why?


## Solution 3

Deletion targets only the first mapping.

Deleting `"mode"` removes it from `child`, so the parent's `"mode"` becomes visible.

Deleting `"y"` through the chain fails because `"y"` is not in the first mapping, even though it is visible through the combined view.


In [5]:
child = {"mode": "test", "x": 1}
parent = {"mode": "prod", "y": 2}
cm = ChainMap(child, parent)

assert cm["mode"] == "test"

del cm["mode"]

assert "mode" not in child
assert cm["mode"] == "prod"

try:
    del cm["y"]
except KeyError as exc:
    print("Expected KeyError:", exc)
else:
    raise AssertionError("Deleting a parent-only key should have failed.")

assert parent["y"] == 2
print("Effective mapping after deletion:", dict(cm))


Expected KeyError: "Key not found in the first mapping: 'y'"
Effective mapping after deletion: {'mode': 'prod', 'y': 2, 'x': 1}


# Problem 4 — Dynamic view behavior

Create a `ChainMap` over two dictionaries. After the `ChainMap` has already been created:

- add a new key to the second dictionary;
- change an existing value in the second dictionary;
- insert a shadowing key in the first dictionary;
- remove the shadowing key again.

Prove with assertions that the chain reflects each change immediately.

### Goal

Demonstrate that `ChainMap` is a live view, not a copied merge.


## Solution 4


In [6]:
front = {"a": 1}
back = {"b": 2, "status": "old"}
cm = ChainMap(front, back)

back["c"] = 3
assert cm["c"] == 3

back["status"] = "new"
assert cm["status"] == "new"

front["status"] = "override"
assert cm["status"] == "override"

del front["status"]
assert cm["status"] == "new"

print("front:", front)
print("back:", back)
print("effective:", dict(cm))


front: {'a': 1}
back: {'b': 2, 'status': 'new', 'c': 3}
effective: {'b': 2, 'status': 'new', 'c': 3, 'a': 1}


# Problem 5 — Scope stack with `new_child()` and `parents`

Model nested variable scopes.

Start with a global scope:

```python
global_scope = {"x": 10, "language": "Python"}
```

Create a local scope with `new_child()` and assign:

```python
x = 20
y = 30
```

Then create an inner scope and assign:

```python
x = 99
z = 40
```

Tasks:

1. Show the effective values in the inner scope.
2. Use `.parents` once to move from inner scope to local scope.
3. Use `.parents` again to move back to global scope.
4. Prove the original global dictionary was not overwritten.


## Solution 5

`new_child()` adds a new mapping at the front. `.parents` returns a new `ChainMap` excluding the current first mapping.

This makes `ChainMap` a natural model for nested lexical scopes.


In [7]:
global_scope = {"x": 10, "language": "Python"}

local_scope = ChainMap(global_scope).new_child()
local_scope["x"] = 20
local_scope["y"] = 30

inner_scope = local_scope.new_child()
inner_scope["x"] = 99
inner_scope["z"] = 40

assert inner_scope["x"] == 99
assert inner_scope["y"] == 30
assert inner_scope["language"] == "Python"
assert inner_scope["z"] == 40

back_to_local = inner_scope.parents
assert back_to_local["x"] == 20
assert back_to_local["y"] == 30

back_to_global = back_to_local.parents
assert back_to_global["x"] == 10
assert "y" not in back_to_global

assert global_scope == {"x": 10, "language": "Python"}

print("Inner maps:")
pprint(inner_scope.maps)
print("Local maps:")
pprint(back_to_local.maps)
print("Global view:")
pprint(back_to_global.maps)


Inner maps:
[{'x': 99, 'z': 40}, {'x': 20, 'y': 30}, {'language': 'Python', 'x': 10}]
Local maps:
[{'x': 20, 'y': 30}, {'language': 'Python', 'x': 10}]
Global view:
[{'language': 'Python', 'x': 10}]


# Problem 6 — Find the provenance of a key

Write a function:

```python
def find_source(chain_map, key):
    ...
```

It should return a tuple:

```python
(index, mapping, value)
```

where:

- `index` is the first mapping index containing the key;
- `mapping` is the actual underlying mapping;
- `value` is the visible value.

If the key does not exist, raise `KeyError`.

### Example

For:

```python
cm = ChainMap(
    {"debug": True},
    {"timeout": 10},
    {"debug": False, "timeout": 30}
)
```

`find_source(cm, "debug")` should report mapping index `0`.

`find_source(cm, "timeout")` should report mapping index `1`.


## Solution 6

Do not flatten the `ChainMap`; provenance information would be lost. Search `cm.maps` directly from left to right.


In [8]:
def find_source(chain_map, key):
    for index, mapping in enumerate(chain_map.maps):
        if key in mapping:
            return index, mapping, mapping[key]
    raise KeyError(key)


cm = ChainMap(
    {"debug": True},
    {"timeout": 10},
    {"debug": False, "timeout": 30},
)

i, mapping, value = find_source(cm, "debug")
assert i == 0
assert value is True

i, mapping, value = find_source(cm, "timeout")
assert i == 1
assert value == 10

try:
    find_source(cm, "missing")
except KeyError:
    pass
else:
    raise AssertionError("Missing key should raise KeyError.")

print(find_source(cm, "debug"))
print(find_source(cm, "timeout"))


(0, {'debug': True}, True)
(1, {'timeout': 10}, 10)


# Problem 7 — Flatten a `ChainMap` while preserving visible precedence

Write:

```python
def flatten_chainmap(cm):
    ...
```

It must return a normal dictionary with exactly the same visible key/value pairs as the `ChainMap`.

Requirements:

- earlier mappings must win;
- the result must be independent from future changes to the original mappings;
- do not mutate any input mapping.

Then prove that later changes to the original dictionaries do not alter the flattened result.


## Solution 7

A direct and readable solution is simply `dict(cm)`, because iteration exposes the effective mapping view.

For learning purposes, an explicit implementation can process mappings from low priority to high priority using `dict.update()`. That means iterating `reversed(cm.maps)`.


In [9]:
def flatten_chainmap(cm):
    result = {}
    for mapping in reversed(cm.maps):
        result.update(mapping)
    return result


high = {"a": 1, "shared": "high"}
middle = {"b": 2, "shared": "middle"}
low = {"c": 3, "shared": "low"}

cm = ChainMap(high, middle, low)
flat = flatten_chainmap(cm)

assert flat == {
    "a": 1,
    "b": 2,
    "c": 3,
    "shared": "high",
}
assert flat == dict(cm)

high["a"] = 999
low["new"] = "later"

assert flat["a"] == 1
assert "new" not in flat

print("Live ChainMap:", dict(cm))
print("Snapshot:", flat)


Live ChainMap: {'c': 3, 'shared': 'high', 'new': 'later', 'b': 2, 'a': 999}
Snapshot: {'c': 3, 'shared': 'high', 'b': 2, 'a': 1}


# Problem 8 — Compare `ChainMap` precedence with dictionary unpacking

Given:

```python
d1 = {"x": 1, "shared": "d1"}
d2 = {"y": 2, "shared": "d2"}
d3 = {"z": 3, "shared": "d3"}
```

Compare:

```python
ChainMap(d1, d2, d3)
```

with:

```python
{**d1, **d2, **d3}
```

Tasks:

1. Predict both values for `"shared"`.
2. Explain why they differ.
3. Construct a dictionary-unpacking expression that matches the visible precedence of `ChainMap(d1, d2, d3)`.


## Solution 8

`ChainMap` uses **first-match wins**.

Dictionary unpacking uses **later assignment wins**.

Therefore, to mimic `ChainMap(d1, d2, d3)` with unpacking, reverse the unpacking order.


In [10]:
d1 = {"x": 1, "shared": "d1"}
d2 = {"y": 2, "shared": "d2"}
d3 = {"z": 3, "shared": "d3"}

cm = ChainMap(d1, d2, d3)
merged_normal = {**d1, **d2, **d3}
merged_like_chainmap = {**d3, **d2, **d1}

assert cm["shared"] == "d1"
assert merged_normal["shared"] == "d3"
assert merged_like_chainmap["shared"] == "d1"
assert merged_like_chainmap == dict(cm)

print("ChainMap visible:", dict(cm))
print("Normal unpacking:", merged_normal)
print("Reversed unpacking:", merged_like_chainmap)


ChainMap visible: {'z': 3, 'shared': 'd1', 'y': 2, 'x': 1}
Normal unpacking: {'x': 1, 'shared': 'd3', 'y': 2, 'z': 3}
Reversed unpacking: {'z': 3, 'shared': 'd1', 'y': 2, 'x': 1}


# Problem 9 — Custom write-through behavior

Standard `ChainMap` writes only to the first mapping.

Create a subclass called `DeepChainMap` with these rules:

- when assigning an existing key, update the **first underlying mapping that already contains the key**;
- when assigning a new key, insert it into the first mapping;
- when deleting a key, delete it from the **first underlying mapping that contains it**;
- raise `KeyError` if deletion finds no matching key.

This is a classic advanced customization exercise because it changes the default mutation policy without changing lookup behavior.


## Solution 9


In [11]:
class DeepChainMap(ChainMap):
    def __setitem__(self, key, value):
        for mapping in self.maps:
            if key in mapping:
                mapping[key] = value
                return
        self.maps[0][key] = value

    def __delitem__(self, key):
        for mapping in self.maps:
            if key in mapping:
                del mapping[key]
                return
        raise KeyError(key)


first = {"a": 1}
second = {"b": 2, "shared": "second"}
third = {"c": 3}

dcm = DeepChainMap(first, second, third)

dcm["b"] = 200
assert "b" not in first
assert second["b"] == 200

dcm["new"] = 999
assert first["new"] == 999

del dcm["c"]
assert "c" not in third

try:
    del dcm["missing"]
except KeyError:
    pass
else:
    raise AssertionError("Expected KeyError for missing key.")

pprint(dcm.maps)


[{'a': 1, 'new': 999}, {'b': 200, 'shared': 'second'}, {}]


# Problem 10 — Safer layered configuration with validation

Design a configuration system with these layers:

1. temporary runtime overrides
2. user configuration
3. environment configuration
4. defaults

Implement:

```python
def build_config(defaults, environment, user, runtime=None):
    ...
```

The function should:

- create a `ChainMap`;
- use a fresh empty runtime mapping if `runtime` is omitted;
- verify that the effective configuration contains:
  - `"host"`
  - `"port"`
  - `"timeout"`
- verify that `port` is an integer in `1..65535`;
- verify that `timeout` is positive;
- return the `ChainMap`.

Then show that modifying the returned configuration does not mutate `defaults`.


## Solution 10


In [12]:
def build_config(defaults, environment, user, runtime=None):
    runtime = {} if runtime is None else runtime
    config = ChainMap(runtime, user, environment, defaults)

    required = ("host", "port", "timeout")
    missing = [key for key in required if key not in config]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")

    port = config["port"]
    timeout = config["timeout"]

    if not isinstance(port, int) or isinstance(port, bool) or not (1 <= port <= 65535):
        raise ValueError("port must be an integer from 1 to 65535")

    if not isinstance(timeout, (int, float)) or isinstance(timeout, bool) or timeout <= 0:
        raise ValueError("timeout must be a positive number")

    return config


defaults = {
    "host": "api.example.com",
    "port": 443,
    "timeout": 30,
    "retries": 3,
}
environment = {"timeout": 15}
user = {"retries": 5}

config = build_config(defaults, environment, user)

assert config["timeout"] == 15
assert config["retries"] == 5

config["host"] = "localhost"

assert config["host"] == "localhost"
assert defaults["host"] == "api.example.com"
assert config.maps[0]["host"] == "localhost"

print(dict(config))
pprint(config.maps)


{'host': 'localhost', 'port': 443, 'timeout': 15, 'retries': 5}
[{'host': 'localhost'},
 {'retries': 5},
 {'timeout': 15},
 {'host': 'api.example.com', 'port': 443, 'retries': 3, 'timeout': 30}]


# Problem 11 — Transactional overlay: commit or rollback

Use a `ChainMap` as a lightweight transaction layer over a base dictionary.

Write these functions:

```python
def begin_transaction(base):
    ...

def commit(transaction):
    ...

def rollback(transaction):
    ...
```

Rules:

- `begin_transaction(base)` returns a `ChainMap` whose first map is a fresh overlay and whose parent is `base`;
- writes affect only the overlay;
- `commit()` applies the overlay to `base` and returns `base`;
- `rollback()` discards the overlay and returns the unchanged base mapping.

For this exercise, deletion of existing base keys is out of scope; focus on inserts and updates.


## Solution 11

This pattern is useful because the overlay can shadow base values without copying the whole base dictionary.


In [13]:
def begin_transaction(base):
    return ChainMap({}, base)

def commit(transaction):
    if not isinstance(transaction, ChainMap) or len(transaction.maps) < 2:
        raise TypeError("Expected a transaction ChainMap with overlay + base")

    overlay = transaction.maps[0]
    base = transaction.maps[1]
    base.update(overlay)
    return base

def rollback(transaction):
    if not isinstance(transaction, ChainMap) or len(transaction.maps) < 2:
        raise TypeError("Expected a transaction ChainMap with overlay + base")
    return transaction.maps[1]


base = {"balance": 100, "status": "open"}

tx = begin_transaction(base)
tx["balance"] = 125
tx["note"] = "pending"

assert base == {"balance": 100, "status": "open"}
assert tx["balance"] == 125

rolled_back = rollback(tx)
assert rolled_back == {"balance": 100, "status": "open"}

tx2 = begin_transaction(base)
tx2["balance"] = 140
tx2["status"] = "review"

committed = commit(tx2)
assert committed["balance"] == 140
assert committed["status"] == "review"

print(base)


{'balance': 140, 'status': 'review'}


# Problem 12 — Symbol table for a tiny interpreter

Implement a small scope manager:

```python
class ScopeStack:
    ...
```

Required operations:

- initialize with a global mapping;
- `push()` adds a new child scope;
- `pop()` removes the current scope, but must not pop past globals;
- `set(name, value)` writes into the current scope;
- `get(name)` performs normal chained lookup;
- `contains_local(name)` checks only the current scope;
- `depth` reports how many active scopes exist, including globals.

Then model:

- global `x = 1`;
- function scope with `x = 2`, `y = 3`;
- block scope with `z = 4`;
- verify lookup and shadowing;
- pop scopes and verify visibility changes.


## Solution 12


In [14]:
class ScopeStack:
    def __init__(self, global_mapping=None):
        if global_mapping is None:
            global_mapping = {}
        self._chain = ChainMap(global_mapping)

    @property
    def depth(self):
        return len(self._chain.maps)

    @property
    def current(self):
        return self._chain.maps[0]

    def push(self):
        self._chain = self._chain.new_child()
        return self

    def pop(self):
        if self.depth == 1:
            raise RuntimeError("Cannot pop the global scope")
        self._chain = self._chain.parents
        return self

    def set(self, name, value):
        self._chain[name] = value

    def get(self, name):
        return self._chain[name]

    def contains_local(self, name):
        return name in self.current

    def snapshot(self):
        return dict(self._chain)


scopes = ScopeStack({"x": 1})
assert scopes.depth == 1
assert scopes.get("x") == 1

scopes.push()
scopes.set("x", 2)
scopes.set("y", 3)

assert scopes.depth == 2
assert scopes.get("x") == 2
assert scopes.get("y") == 3
assert scopes.contains_local("x")

scopes.push()
scopes.set("z", 4)

assert scopes.depth == 3
assert scopes.get("x") == 2
assert scopes.get("y") == 3
assert scopes.get("z") == 4
assert not scopes.contains_local("x")

scopes.pop()
assert scopes.depth == 2

scopes.pop()
assert scopes.depth == 1
assert scopes.get("x") == 1

try:
    scopes.get("y")
except KeyError:
    pass
else:
    raise AssertionError("y should no longer be visible.")

try:
    scopes.pop()
except RuntimeError as exc:
    print("Expected:", exc)
else:
    raise AssertionError("Should not pop global scope.")

print(scopes.snapshot())


Expected: Cannot pop the global scope
{'x': 1}


# Problem 13 — Immutable parent mappings

Suppose application defaults must never be mutated accidentally.

Use `MappingProxyType` to make the defaults read-only, then place a normal mutable overlay in front:

```python
defaults = MappingProxyType({...})
config = ChainMap({}, defaults)
```

Tasks:

1. override a default through the `ChainMap`;
2. prove the defaults remain unchanged;
3. attempt direct mutation of the proxy and observe the failure;
4. explain why placing the immutable mapping first would make normal `ChainMap` assignment fail.


## Solution 13


In [15]:
raw_defaults = {
    "theme": "light",
    "timeout": 30,
}
defaults = MappingProxyType(raw_defaults)

config = ChainMap({}, defaults)
config["theme"] = "dark"

assert config["theme"] == "dark"
assert defaults["theme"] == "light"
assert config.maps[0]["theme"] == "dark"

try:
    defaults["theme"] = "blue"
except TypeError as exc:
    print("Expected TypeError from MappingProxyType:", exc)
else:
    raise AssertionError("MappingProxyType should be immutable.")

bad_order = ChainMap(defaults, {})
try:
    bad_order["x"] = 1
except TypeError as exc:
    print("Expected write failure when immutable map is first:", exc)
else:
    raise AssertionError("Write should fail because maps[0] is immutable.")


Expected TypeError from MappingProxyType: 'mappingproxy' object does not support item assignment
Expected write failure when immutable map is first: 'mappingproxy' object does not support item assignment


# Problem 14 — Nested `ChainMap` vs flat `ChainMap`

Compare:

```python
nested = ChainMap(ChainMap(a, b), c)
flat = ChainMap(a, b, c)
```

Tasks:

1. determine whether lookups produce the same visible values;
2. inspect `.maps` for both;
3. show that their structures are different;
4. write a helper that recursively flattens nested `ChainMap` objects into a single `ChainMap` with the same left-to-right lookup order.


## Solution 14

Nested `ChainMap` objects can behave similarly for lookup, but the first element of `nested.maps` is itself a `ChainMap`. Flattening the structure can make inspection and custom mutation policies simpler.


In [16]:
def flatten_chainmap_structure(cm):
    flattened = []

    def visit(mapping):
        if isinstance(mapping, ChainMap):
            for child in mapping.maps:
                visit(child)
        else:
            flattened.append(mapping)

    visit(cm)
    return ChainMap(*flattened)


a = {"a": 1, "shared": "a"}
b = {"b": 2, "shared": "b"}
c = {"c": 3, "shared": "c"}

nested = ChainMap(ChainMap(a, b), c)
flat = ChainMap(a, b, c)
normalized = flatten_chainmap_structure(nested)

assert nested["shared"] == "a"
assert flat["shared"] == "a"
assert dict(nested) == dict(flat) == dict(normalized)

assert isinstance(nested.maps[0], ChainMap)
assert not isinstance(flat.maps[0], ChainMap)

assert normalized.maps == [a, b, c]

print("nested.maps:")
pprint(nested.maps)
print("normalized.maps:")
pprint(normalized.maps)


nested.maps:
[ChainMap({'a': 1, 'shared': 'a'}, {'b': 2, 'shared': 'b'}),
 {'c': 3, 'shared': 'c'}]
normalized.maps:
[{'a': 1, 'shared': 'a'}, {'b': 2, 'shared': 'b'}, {'c': 3, 'shared': 'c'}]


# Problem 15 — Effective differences between two layered configurations

Write:

```python
def effective_diff(left, right):
    ...
```

where `left` and `right` can be any mapping objects, including `ChainMap`.

Return a dictionary of changed keys in this form:

```python
{
    key: (left_value_or_missing, right_value_or_missing)
}
```

Use a unique sentinel object for missing keys so that a real value of `None` is not confused with absence.

The function should compare the **effective visible mapping**, not each layer separately.


## Solution 15


In [17]:
MISSING = object()

def effective_diff(left, right):
    keys = set(left) | set(right)
    result = {}

    for key in keys:
        left_value = left[key] if key in left else MISSING
        right_value = right[key] if key in right else MISSING

        if left_value != right_value:
            result[key] = (left_value, right_value)

    return result


defaults = {"host": "prod", "port": 80, "debug": False}
left = ChainMap({"debug": True}, defaults)
right = ChainMap({"port": 8080, "extra": None}, defaults)

diff = effective_diff(left, right)

assert diff["debug"] == (True, False)
assert diff["port"] == (80, 8080)
assert diff["extra"][0] is MISSING
assert diff["extra"][1] is None
assert "host" not in diff

for key, (old, new) in diff.items():
    old_display = "<MISSING>" if old is MISSING else old
    new_display = "<MISSING>" if new is MISSING else new
    print(f"{key!r}: {old_display!r} -> {new_display!r}")


'extra': '<MISSING>' -> None
'port': 80 -> 8080
'debug': True -> False


# Problem 16 — Selective commit from an override layer

You have a temporary overlay that contains several experimental values:

```python
base = {"timeout": 30, "retries": 3, "theme": "light"}
overlay = {"timeout": 5, "retries": 10, "theme": "dark", "debug": True}
cm = ChainMap(overlay, base)
```

Write:

```python
def commit_keys(cm, keys):
    ...
```

that copies only selected keys from `cm.maps[0]` into `cm.maps[1]`.

Requirements:

- only keys physically present in the overlay may be committed;
- missing requested keys should raise `KeyError`;
- after committing, remove those keys from the overlay so that the base value becomes the source;
- return the chain map.

Commit only `"timeout"` and `"theme"`.


## Solution 16


In [18]:
def commit_keys(cm, keys):
    if len(cm.maps) < 2:
        raise ValueError("Need at least overlay and base mappings")

    overlay = cm.maps[0]
    base = cm.maps[1]

    keys = list(keys)
    missing = [key for key in keys if key not in overlay]
    if missing:
        raise KeyError(f"Not present in overlay: {missing}")

    for key in keys:
        base[key] = overlay[key]

    for key in keys:
        del overlay[key]

    return cm


base = {"timeout": 30, "retries": 3, "theme": "light"}
overlay = {"timeout": 5, "retries": 10, "theme": "dark", "debug": True}
cm = ChainMap(overlay, base)

commit_keys(cm, ["timeout", "theme"])

assert base["timeout"] == 5
assert base["theme"] == "dark"
assert "timeout" not in overlay
assert "theme" not in overlay

assert cm["timeout"] == 5
assert cm["theme"] == "dark"
assert cm["retries"] == 10
assert cm["debug"] is True

print("base:", base)
print("overlay:", overlay)
print("effective:", dict(cm))


base: {'timeout': 5, 'retries': 3, 'theme': 'dark'}
overlay: {'retries': 10, 'debug': True}
effective: {'timeout': 5, 'retries': 10, 'theme': 'dark', 'debug': True}


# Problem 17 — Provenance report for every effective key

Write:

```python
def provenance_report(cm):
    ...
```

Return a dictionary whose values contain:

- the visible value;
- the source mapping index;
- whether the key is shadowed by duplicate occurrences deeper in the chain;
- all mapping indexes that contain the key.

Example shape:

```python
{
    "timeout": {
        "value": 5,
        "source_index": 0,
        "shadowed": True,
        "present_in": [0, 2]
    }
}
```

This is useful for debugging complex configuration stacks.


## Solution 17


In [19]:
def provenance_report(cm):
    locations = {}

    for index, mapping in enumerate(cm.maps):
        for key in mapping:
            locations.setdefault(key, []).append(index)

    report = {}

    for key, indexes in locations.items():
        source_index = indexes[0]
        report[key] = {
            "value": cm.maps[source_index][key],
            "source_index": source_index,
            "shadowed": len(indexes) > 1,
            "present_in": indexes,
        }

    return report


cm = ChainMap(
    {"timeout": 5, "debug": True},
    {"theme": "dark"},
    {"timeout": 30, "debug": False, "theme": "light"},
)

report = provenance_report(cm)

assert report["timeout"]["value"] == 5
assert report["timeout"]["source_index"] == 0
assert report["timeout"]["shadowed"] is True
assert report["timeout"]["present_in"] == [0, 2]

assert report["theme"]["value"] == "dark"
assert report["theme"]["present_in"] == [1, 2]

pprint(report)


{'debug': {'present_in': [0, 2],
           'shadowed': True,
           'source_index': 0,
           'value': True},
 'theme': {'present_in': [1, 2],
           'shadowed': True,
           'source_index': 1,
           'value': 'dark'},
 'timeout': {'present_in': [0, 2],
             'shadowed': True,
             'source_index': 0,
             'value': 5}}


# Problem 18 — Lightweight performance experiment

Compare lookup speed for:

1. a `ChainMap` with several layers;
2. a fully merged dictionary snapshot.

Do separate timings for:

- a key found in the first layer;
- a key found in the last layer.

Do **not** assume the result is universal. The goal is to reason about trade-offs:

- `ChainMap` avoids copying when constructed;
- deeper lookups may inspect several mappings;
- a materialized dictionary pays merge/copy cost up front but has normal dictionary lookup afterward.


## Solution 18

The exact timings depend on Python version, hardware, dictionary sizes, and benchmark setup. Focus on the relative pattern rather than exact numbers.


In [20]:
layers = [
    {"first": 1},
    {"k1": 1},
    {"k2": 2},
    {"k3": 3},
    {"last": 999},
]

cm = ChainMap(*layers)

snapshot = {}
for mapping in reversed(layers):
    snapshot.update(mapping)

assert cm["first"] == snapshot["first"]
assert cm["last"] == snapshot["last"]

repeat = 200_000

cm_first = timeit.timeit(lambda: cm["first"], number=repeat)
cm_last = timeit.timeit(lambda: cm["last"], number=repeat)
dict_first = timeit.timeit(lambda: snapshot["first"], number=repeat)
dict_last = timeit.timeit(lambda: snapshot["last"], number=repeat)

print(f"ChainMap first-layer lookup: {cm_first:.6f}s")
print(f"ChainMap last-layer lookup:  {cm_last:.6f}s")
print(f"dict first-key lookup:       {dict_first:.6f}s")
print(f"dict last-key lookup:        {dict_last:.6f}s")

print("\nInterpretation:")
print("- ChainMap construction is cheap because mappings are referenced, not merged.")
print("- Deeper ChainMap lookups may cost more because mappings are searched in order.")
print("- A normal dict snapshot usually gives direct dict lookup after paying merge cost.")


ChainMap first-layer lookup: 0.050718s
ChainMap last-layer lookup:  0.418268s
dict first-key lookup:       0.010241s
dict last-key lookup:        0.010345s

Interpretation:
- ChainMap construction is cheap because mappings are referenced, not merged.
- Deeper ChainMap lookups may cost more because mappings are searched in order.
- A normal dict snapshot usually gives direct dict lookup after paying merge cost.


# Challenge Problem 19 — Masking deletions with a tombstone layer

Standard `ChainMap` cannot represent "this key is deleted from the effective configuration while still existing in a parent" unless you physically delete from a parent or customize lookup behavior.

Implement a mutable mapping called `MaskingChainMap` that supports:

- layered lookup like `ChainMap`;
- writes into the first mapping;
- deletion that **masks** a key from all parent mappings without mutating them;
- assigning a key again should unmask it;
- a key masked in the child should behave as absent even if parents contain it.

Hint: use a private sentinel object or a dedicated set of tombstoned keys.

This models overlays used in package managers, filesystem unions, and configuration systems where "delete" should hide a parent value.


## Solution 19

One clean design keeps a set of masked keys. Lookup first checks whether a key is masked; if so, it behaves as missing. Assignment removes any mask and writes to the first mapping.


In [21]:
class MaskingChainMap(MutableMapping):
    def __init__(self, *maps):
        self.maps = list(maps) if maps else [{}]
        self._masked = set()

    def __getitem__(self, key):
        if key in self._masked:
            raise KeyError(key)

        for mapping in self.maps:
            if key in mapping:
                return mapping[key]

        raise KeyError(key)

    def __setitem__(self, key, value):
        self._masked.discard(key)
        self.maps[0][key] = value

    def __delitem__(self, key):
        # Deleting a key means "hide it from the effective view".
        if key not in self:
            raise KeyError(key)

        self.maps[0].pop(key, None)
        self._masked.add(key)

    def __iter__(self):
        seen = set()

        for mapping in self.maps:
            for key in mapping:
                if key not in seen and key not in self._masked:
                    seen.add(key)
                    yield key

    def __len__(self):
        return sum(1 for _ in self)

    def unmask(self, key):
        self._masked.discard(key)


child = {}
parent = {"a": 1, "b": 2}

mcm = MaskingChainMap(child, parent)

assert mcm["a"] == 1
assert "a" in mcm

del mcm["a"]

assert "a" not in mcm
assert parent["a"] == 1

try:
    mcm["a"]
except KeyError:
    pass
else:
    raise AssertionError("Masked key should be invisible.")

mcm["a"] = 100

assert mcm["a"] == 100
assert child["a"] == 100
assert parent["a"] == 1

print(dict(mcm))
print("masked:", mcm._masked)


{'a': 100, 'b': 2}
masked: set()


# Integrated Case Study — Production-style configuration stack

Build a realistic configuration pipeline with:

- immutable application defaults;
- environment settings;
- user settings;
- request-level overrides;
- provenance reporting;
- validation;
- temporary experimental scope;
- selective commit;
- final snapshot for serialization.

The goal is to combine the previous patterns into one coherent workflow.


In [22]:
# 1) Immutable defaults
raw_defaults = {
    "host": "api.example.com",
    "port": 443,
    "timeout": 30,
    "retries": 3,
    "theme": "light",
}
defaults = MappingProxyType(raw_defaults)

# 2) Lower-priority mutable layers
environment = {
    "timeout": 20,
    "retries": 5,
}
user = {
    "theme": "dark",
}
request = {
    "timeout": 4,
}

# 3) Compose
config = ChainMap(request, user, environment, defaults)

assert config["host"] == "api.example.com"
assert config["port"] == 443
assert config["timeout"] == 4
assert config["retries"] == 5
assert config["theme"] == "dark"

# 4) Provenance
report = provenance_report(config)
assert report["timeout"]["source_index"] == 0
assert report["theme"]["source_index"] == 1
assert report["retries"]["source_index"] == 2
assert report["host"]["source_index"] == 3

# 5) Add a temporary experiment scope
experiment = config.new_child()
experiment["timeout"] = 1
experiment["feature_x"] = True

assert experiment["timeout"] == 1
assert config["timeout"] == 4
assert "feature_x" not in config

# 6) Roll back simply by dropping the child scope
rolled_back = experiment.parents

assert rolled_back["timeout"] == 4
assert "feature_x" not in rolled_back

# 7) Final serialization snapshot
final_snapshot = dict(config)

assert final_snapshot == {
    "host": "api.example.com",
    "port": 443,
    "timeout": 4,
    "retries": 5,
    "theme": "dark",
}

pprint(final_snapshot)


{'host': 'api.example.com',
 'port': 443,
 'retries': 5,
 'theme': 'dark',
 'timeout': 4}


# Additional Drill Questions

Use these as extra exercises. Solutions can be built from the techniques above.

1. Write `count_shadowed_keys(cm)` that counts keys appearing in more than one underlying mapping.
2. Write `all_values_for_key(cm, key)` returning every occurrence from left to right.
3. Write `reorder_layer(cm, old_index, new_index)` by manipulating `cm.maps`.
4. Build a function that inserts an emergency override mapping at index `0`.
5. Build a read-only public view by wrapping `dict(cm)` in `MappingProxyType`.
6. Write a function that returns only keys originating from the first mapping.
7. Write a function that returns keys visible from parents but currently shadowed by the child.
8. Implement a context manager that temporarily pushes a child scope and automatically discards it.
9. Compare `ChainMap` with repeated `dict.update()` for a 20-layer configuration.
10. Modify `DeepChainMap` so writes to selected protected keys are forbidden.
11. Add type validation to `ScopeStack.set()`.
12. Add a `source_of(name)` method to `ScopeStack`.
13. Write a recursive pretty-printer for nested `ChainMap` objects.
14. Write tests proving that `dict(cm)` is a snapshot while `cm` remains live.
15. Create a layered command-line argument configuration pattern:
    `CLI -> environment -> config file -> defaults`.


# Summary

Key advanced ideas:

- `ChainMap` is a **live composite view**, not a merged copy.
- lookup precedence is left-to-right / first-match-wins.
- standard writes and deletions target only `maps[0]`.
- deleting a child key can reveal a parent key.
- `new_child()` and `parents` model nested scopes elegantly.
- `.maps` exposes the actual ordered mapping list.
- flattening creates a snapshot and loses live-reference behavior.
- custom subclasses can implement alternate write/delete policies.
- immutable parents plus a mutable overlay are a strong configuration pattern.
- provenance tools are useful for debugging large configuration stacks.
- overlays can model transactions, temporary overrides, and scope stacks.
